In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Armaan457/Cyber_Qwen",
    max_seq_length=512,
    load_in_4bit=False,
    device_map="auto",
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)

In [ ]:
import pyarrow.parquet as pq
table = pq.read_table("/content/test-00000-of-00001.parquet")
test_df = table.to_pandas()
test_df.columns

Index(['instruction', 'input', 'output', 'source', 'persona', 'perspective',
       'category'],
      dtype='object')

In [ ]:
import pandas as pd

SYSTEM_PROMPT = (
    "You are CyberCTRL, an expert cybersecurity assistant. "
    "Provide technically correct, clear, and complete answers."
)

def create_messages(row):
    user_prompt = row["instruction"].strip()
    if pd.notna(row["input"]) and row["input"].strip():
        user_prompt += "\n\n" + row["input"].strip()
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
        {
            "role": "assistant",
            "content": row["output"].strip(),
        },
    ]

test_df["messages"] = test_df.apply(create_messages, axis=1)

In [ ]:
from datasets import Dataset

test_ds = Dataset.from_pandas(test_df, preserve_index=False)

def formatting(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

test_ds = test_ds.map(formatting)

Map:   0%|          | 0/19310 [00:00<?, ? examples/s]

In [ ]:
test_ds = test_ds.filter(
    lambda x: len(tokenizer(x["text"]).input_ids) <= 512
)

Filter:   0%|          | 0/19310 [00:00<?, ? examples/s]

In [ ]:
num_examples = min(5000, len(test_ds))
test_ds = test_ds.shuffle(seed=42).select(range(num_examples))

In [ ]:
from tqdm import tqdm
import json
import os
import pandas as pd
import torch

batch_size = 8
checkpoint_path = "CyberCTRL_eval_checkpoint.json"
predictions_path = "CyberCTRL_predictions.csv"
resume = True

if resume and os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    predictions = checkpoint["predictions"]
    references = checkpoint["references"]
    print(f"Resuming from {len(predictions)} completed examples.")
else:
    predictions = []
    references = []

if len(predictions) != len(references):
    raise ValueError("Checkpoint predictions and references have different lengths.")

if len(predictions) > len(test_ds):
    raise ValueError("Checkpoint contains more examples than the current dataset.")

model.generation_config.max_length = None

for i in tqdm(range(len(predictions), len(test_ds), batch_size)):
    batch = test_ds[i:i+batch_size]
    prompts = [
        tokenizer.apply_chat_template(
            sample[:-1],
            tokenize=False,
            add_generation_prompt=True,
        )
        for sample in batch["messages"]
    ]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]

    for output in outputs:
        pred = tokenizer.decode(
            output[prompt_len:],
            skip_special_tokens=True,
        ).strip()
        predictions.append(pred)

    references.extend(
        msg[-1]["content"].strip()
        for msg in batch["messages"]
    )

    checkpoint = {
        "predictions": predictions,
        "references": references,
    }
    temp_checkpoint_path = checkpoint_path + ".tmp"
    with open(temp_checkpoint_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False)
    os.replace(temp_checkpoint_path, checkpoint_path)

    pd.DataFrame({
        "reference": references,
        "prediction": predictions,
    }).to_csv(predictions_path, index=False)

print(f"Saved {len(predictions)} predictions to {predictions_path}.")

Resuming from 2784 completed examples.


100%|██████████| 277/277 [1:03:42<00:00, 13.80s/it]

Saved 5000 predictions to CyberCTRL_predictions.csv.


In [18]:
from evaluate import load
import numpy as np

rouge = load("rouge")
bertscore = load("bertscore")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
)

bert = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    device="cuda",
)

exact_match = np.mean([
    p == r
    for p, r in zip(predictions, references)
])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
print("ROUGE")
print()
for k, v in rouge_scores.items():
    print(f"{k:10s}: {v:.4f}")

ROUGE

rouge1    : 0.6363
rouge2    : 0.5679
rougeL    : 0.6226
rougeLsum : 0.6253


In [20]:
print("BERTScore")
print()
print(f"Precision : {np.mean(bert['precision']):.4f}")
print(f"Recall    : {np.mean(bert['recall']):.4f}")
print(f"F1        : {np.mean(bert['f1']):.4f}")

BERTScore

Precision : 0.9193
Recall    : 0.9178
F1        : 0.9183


In [22]:
pd.DataFrame({
    "reference": references,
    "prediction": predictions,
}).to_csv("CyberCTRL_predictions.csv", index=False)